# Coleta, Integração e Limpeza dos Dados — DIEESE (Cesta Básica)

Este notebook documenta, de forma reprodutível, todo o processo de construção da base de dados usada no projeto:

1. **Coleta** dos boletins mensais da Pesquisa Nacional da Cesta Básica (DIEESE), a partir da página de índice de boletins anteriores.
2. **Correção pontual** de um boletim cujo link estava mapeado incorretamente.
3. **Limpeza dos arquivos HTML** que na verdade eram apenas páginas "casca" contendo o link real do PDF.
4. **Extração das tabelas dos PDFs** (parsing das linhas de cada capital) e consolidação em uma única base.
5. **Correção manual** de um mês cujo PDF não pôde ser processado automaticamente.
6. **Exportação da base tratada final** (`base_tratada_dieese_completa.csv`).

**Fonte dos dados:** [DIEESE — Análise da Cesta Básica](https://www.dieese.org.br/analisecestabasica/analiseCestaBasicaAnteriores.html)

> ⚠️ Este notebook faz requisições HTTP reais ao site do DIEESE e respeita um intervalo de 10 segundos entre downloads (conforme o `Crawl-delay` do `robots.txt` do site). Rodar o notebook do zero, portanto, pode levar bastante tempo dependendo de quantos boletins ainda não estiverem salvos localmente em `dados_brutos/dieese/`.


## 1. Configuração e bibliotecas

Todas as bibliotecas usadas ao longo do notebook (coleta, parsing de HTML, parsing de PDF e manipulação tabular) são importadas aqui, uma única vez.

In [ ]:
# --- bibliotecas de coleta e manipulação de arquivos ---
import requests, time, re, csv, os
from pathlib import Path
from datetime import datetime
from bs4 import BeautifulSoup

# --- bibliotecas de extração de PDF e manipulação tabular ---
import pdfplumber
import pandas as pd


## 2. Coleta dos boletins do DIEESE

Nesta etapa:
- Acessamos a página de índice com o histórico de boletins da Cesta Básica.
- Identificamos, via expressão regular, os links que correspondem a boletins mensais (padrão `cestabasica` + `AAMMES`).
- Baixamos cada boletim (HTML ou PDF, conforme disponibilizado pelo site) para `dados_brutos/dieese/`.
- Registramos a proveniência de cada arquivo (URL, status HTTP, formato e data/hora da coleta) em `dados_brutos/dieese/provenance_log.csv`.
- Se um arquivo já existir em disco (execução anterior do notebook), ele **não é baixado de novo** — apenas é registrado no log com a data de modificação do arquivo já existente, evitando re-coleta desnecessária.


In [ ]:
BASE = "https://www.dieese.org.br"
INDEX_URL = f"{BASE}/analisecestabasica/analiseCestaBasicaAnteriores.html"
HEADERS = {"User-Agent": "Projeto-UFAM/1.0"}

# pasta onde os dados brutos (crus, exatamente como baixados) ficam salvos
Path("dados_brutos/dieese").mkdir(parents=True, exist_ok=True)

# log de proveniência: registra de onde/quando/como cada arquivo foi coletado
caminho_log = "dados_brutos/dieese/provenance_log.csv"
campos = ["aammes", "titulo", "url", "status_http", "formato", "coletado_em"]

arquivo_novo = not Path(caminho_log).exists()
arquivo_log = open(caminho_log, "a", newline="", encoding="utf-8")   # "a" = adicionar, não sobrescrever
escritor = csv.DictWriter(arquivo_log, fieldnames=campos)
if arquivo_novo:
    escritor.writeheader()


In [ ]:
# baixa a página de índice e extrai todos os links que apontam para boletins mensais
resposta = requests.get(INDEX_URL, headers=HEADERS, timeout=30)
sopa = BeautifulSoup(resposta.text, "html.parser")

# padrão que identifica boletins pelo nome do arquivo (ex: cestabasica202001 ou 202001cestabasica)
padrao = re.compile(r"(cestabasica\d{6}|\d{6}cestabasica)", re.IGNORECASE)
boletins = [(a.get_text(strip=True), a["href"]) for a in sopa.find_all("a", href=True) if padrao.search(a["href"])]

print("Total de boletins encontrados:", len(boletins))


In [ ]:
# baixa cada boletim identificado, respeitando o intervalo de 10s entre requisições
for titulo, href in boletins:
    url = href if href.startswith("http") else BASE + href
    extensao = url.split(".")[-1]
    aammes = re.search(r"(\d{6})", url).group(1)

    caminho_html = f"dados_brutos/dieese/{aammes}.html"
    caminho_pdf = f"dados_brutos/dieese/{aammes}.pdf"

    # se já existe, apenas registra no log usando a data do arquivo (baixado em execução anterior)
    if Path(caminho_html).exists() or Path(caminho_pdf).exists():
        existente = caminho_html if Path(caminho_html).exists() else caminho_pdf
        hora = datetime.fromtimestamp(os.path.getmtime(existente)).strftime("%Y-%m-%d %H:%M:%S")
        escritor.writerow({"aammes": aammes, "titulo": titulo, "url": url, "status_http": 200,
                            "formato": existente.split(".")[-1],
                            "coletado_em": hora + " (recuperado, já existia no disco)"})
        arquivo_log.flush()
        print(aammes, "-> já existia, registrado no log")
        continue

    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code == 200:
            with open(f"dados_brutos/dieese/{aammes}.{extensao}", "wb") as f:
                f.write(r.content)
            print(aammes, "-> salvo como", extensao)
        else:
            print(aammes, "-> falhou, status", r.status_code)
        escritor.writerow({"aammes": aammes, "titulo": titulo, "url": url,
                            "status_http": r.status_code, "formato": extensao,
                            "coletado_em": time.strftime("%Y-%m-%d %H:%M:%S")})
    except requests.exceptions.RequestException as erro:
        print(aammes, "-> erro de conexão, pulando:", erro)
        escritor.writerow({"aammes": aammes, "titulo": titulo, "url": url,
                            "status_http": "erro", "formato": extensao,
                            "coletado_em": time.strftime("%Y-%m-%d %H:%M:%S")})

    arquivo_log.flush()   # grava no log imediatamente, sem esperar o loop inteiro terminar
    time.sleep(10)        # respeita o Crawl-delay do robots.txt do site

arquivo_log.close()
print("Coleta finalizada.")


### 2.1 Correção pontual: boletim de janeiro/2010

O boletim de `201001` estava mapeado no índice do site com uma URL que aponta para um HTML, mas o conteúdo real é um PDF hospedado em outro caminho (`/analisecestabasica/2009/201001cestabasica.pdf`). Esta célula corrige esse caso específico: baixa o PDF correto e remove o `.html` incorreto que porventura tenha sido salvo no lugar.


In [ ]:
url_201001 = "https://www.dieese.org.br/analisecestabasica/2009/201001cestabasica.pdf"
r = requests.get(url_201001, headers={"User-Agent": "Projeto-UFAM/1.0"}, timeout=30)

if r.status_code == 200:
    Path("dados_brutos/dieese/201001.pdf").write_bytes(r.content)
    Path("dados_brutos/dieese/201001.html").unlink(missing_ok=True)  # remove a "casca" antiga, se existir
    print("Boletim 201001 salvo com sucesso!")
else:
    print("Ainda deu erro, status:", r.status_code)


## 3. Limpeza dos arquivos HTML ("cascas")

Alguns boletins foram salvos como `.html`, mas na verdade são páginas intermediárias (uma "casca") que apenas exibem um link para o PDF real dos "Resultados Mensais". Esta etapa:

1. Varre todos os `.html` já baixados em `dados_brutos/dieese/`.
2. Para cada um, verifica se é de fato uma página-casca (contém o texto `"Resultados Mensais de"`).
3. Se for, extrai o link do PDF real embutido nela, baixa o PDF e **substitui** o `.html` pelo `.pdf` correspondente.
4. HTMLs que não são cascas (conteúdo de verdade) são mantidos como estão.


In [ ]:
BASE = "https://www.dieese.org.br"
HEADERS = {"User-Agent": "Projeto-UFAM/1.0"}
pasta = Path("dados_brutos/dieese")

def achar_pdf_real(caminho_html):
    """Se o .html for uma página-casca, devolve o link do PDF real escondido nela."""
    texto = caminho_html.read_text(encoding="iso-8859-1", errors="ignore")
    if "Resultados Mensais de" not in texto:
        return None
    m = re.search(r'href="([^"]*cestabasica\.pdf)"', texto)
    if m:
        href = m.group(1)
        return href if href.startswith("http") else BASE + href
    return None


In [ ]:
arquivos_html = sorted(pasta.glob("*.html"))
print("Total de .html encontrados:", len(arquivos_html))

convertidos, mantidos, falharam = [], [], []

for arquivo_html in arquivos_html:
    link_pdf = achar_pdf_real(arquivo_html)
    if link_pdf is None:
        mantidos.append(arquivo_html.name)
        continue

    r = requests.get(link_pdf, headers=HEADERS, timeout=30)
    if r.status_code == 200:
        novo_caminho = arquivo_html.with_suffix(".pdf")
        novo_caminho.write_bytes(r.content)
        arquivo_html.unlink()  # remove a casca, já que o PDF real tomou o lugar dela
        convertidos.append(arquivo_html.name)
        print(arquivo_html.name, "-> convertido para", novo_caminho.name)
    else:
        falharam.append((arquivo_html.name, r.status_code))
        print(arquivo_html.name, "-> falhou ao baixar PDF, status", r.status_code)

    time.sleep(10)  # respeita o Crawl-delay do robots.txt

print("\nConvertidos para PDF:", len(convertidos))
print("Mantidos como HTML de verdade:", len(mantidos), mantidos)
print("Falharam:", falharam if falharam else "nenhum")


## 4. Extração das tabelas dos PDFs

Cada boletim em PDF traz uma tabela com uma linha por capital (valor da cesta básica, tempo de trabalho necessário para comprá-la, variações mensal/anual/12 meses etc.). Como o layout varia um pouco entre os meses, a extração é feita por **posição relativa dos campos na linha**, não por ordem fixa de colunas:

- O campo `tempo` é identificado pelo formato `XhYm`.
- O campo `valor` é o único número maior que 100.
- `pct_sal` e `var_mensal` são os campos vizinhos de `tempo` e `valor`, respectivamente.
- O que sobra são as variações `var_ano` e `var_12` (quando presentes).

As funções abaixo implementam esse parser linha a linha e depois percorrem todos os PDFs baixados, concatenando o resultado em uma única base.


In [ ]:
def normalizar_tempo(texto):
    """Troca '111h 57min' por '111h57m' (formato único, compacto)."""
    return re.sub(r'(\d+)h\s*(\d+)\s*(?:min|m)\b', r'\1h\2m', texto)

def numero(t):
    """Converte texto tipo '1.234,56' em float. '(---)' vira None (dado inexistente)."""
    if t in ("NA", "(---)"):
        return None
    t = t.rstrip('%')
    try:
        return float(t.replace(',', '.'))
    except ValueError:
        return None

def parse_linha_generico(linha):
    """Reconhece cada campo pela posição relativa aos outros, não pela ordem fixa.
    Regra: 'tempo' tem o formato Xh Ym; 'valor' é o único número > 100; 'pct_sal'
    fica sempre colado no 'tempo'; 'var_mensal' fica sempre colado no 'valor'."""
    linha = normalizar_tempo(linha.strip())
    m_capital = re.match(r'^([^\d(\-]+)\s*(?:\(\d+\)\s*)?(.*)$', linha)
    if not m_capital:
        return None
    capital, resto = m_capital.groups()
    capital = capital.strip()
    tokens = resto.split()
    if len(tokens) < 4:
        return None

    idx_tempo = next((i for i, t in enumerate(tokens) if re.match(r'^\d+h\d+m$', t)), None)
    if idx_tempo is None:
        return None
    idx_valor = next((i for i, t in enumerate(tokens) if i != idx_tempo and (numero(t) or 0) > 100), None)
    if idx_valor is None:
        return None

    vizinhos_tempo = [i for i in (idx_tempo - 1, idx_tempo + 1) if 0 <= i < len(tokens) and i != idx_valor]
    idx_pct = vizinhos_tempo[0] if vizinhos_tempo else None
    vizinhos_valor = [i for i in (idx_valor - 1, idx_valor + 1) if 0 <= i < len(tokens) and i not in (idx_tempo, idx_pct)]
    idx_var_mensal = vizinhos_valor[0] if vizinhos_valor else None
    if idx_pct is None or idx_var_mensal is None:
        return None

    usados = {idx_tempo, idx_valor, idx_pct, idx_var_mensal}
    sobra = [i for i in range(len(tokens)) if i not in usados]
    var_ano, var_12 = None, None
    if len(sobra) == 1:
        if sobra[0] < idx_valor:
            var_ano = numero(tokens[sobra[0]])
        else:
            var_12 = numero(tokens[sobra[0]])
    elif len(sobra) >= 2:
        var_ano = numero(tokens[sobra[0]])
        var_12 = numero(tokens[sobra[1]])

    return {"capital": capital, "valor": numero(tokens[idx_valor]), "var_mensal": numero(tokens[idx_var_mensal]),
            "pct_sal": numero(tokens[idx_pct]), "tempo": tokens[idx_tempo], "var_ano": var_ano, "var_12": var_12}


In [ ]:
def linhas_da_pagina(pagina, tolerancia=3):
    """Junta palavras na mesma linha visual, mesmo com pequenas variações de posição (sub-pixel)."""
    palavras = sorted(pagina.extract_words(x_tolerance=2, y_tolerance=3), key=lambda w: w["top"])
    linhas, atual, ultimo_top = [], [], None
    for p in palavras:
        if ultimo_top is not None and p["top"] - ultimo_top > tolerancia:
            linhas.append(atual)
            atual = []
        atual.append(p)
        ultimo_top = p["top"]
    if atual:
        linhas.append(atual)
    return [" ".join(w["text"] for w in sorted(g, key=lambda w: w["x0"])) for g in linhas]

def linhas_validas_da_pagina(pagina):
    return [c for c in (parse_linha_generico(l) for l in linhas_da_pagina(pagina)) if c]

def extrair_tabela_pdf(caminho_arquivo):
    """Percorre as páginas do PDF até achar a que contém a tabela de capitais
    (heurística: pelo menos 9 linhas reconhecidas). Se a tabela continuar na
    página seguinte, junta as linhas novas também."""
    with pdfplumber.open(caminho_arquivo) as pdf:
        paginas = pdf.pages
        for i, pagina in enumerate(paginas):
            candidatos = linhas_validas_da_pagina(pagina)
            if len(candidatos) >= 9:
                if i + 1 < len(paginas):
                    ja_achadas = {c["capital"] for c in candidatos}
                    continuacao = linhas_validas_da_pagina(paginas[i + 1])
                    novos = [c for c in continuacao if c["capital"] not in ja_achadas]
                    if 0 < len(novos) <= 15:
                        candidatos += novos
                df = pd.DataFrame(candidatos)
                df["aammes"] = Path(caminho_arquivo).stem
                return df
    return None


In [ ]:
# roda a extração em todos os PDFs baixados
pasta = Path("dados_brutos/dieese")
arquivos_pdf = sorted(pasta.glob("*.pdf"))
print("Total de PDFs encontrados:", len(arquivos_pdf))

tabelas, falharam = [], []
for caminho in arquivos_pdf:
    df = extrair_tabela_pdf(str(caminho))
    (tabelas if df is not None else falharam).append(df if df is not None else caminho.name)

base_pdf = pd.concat(tabelas, ignore_index=True)
print("\nTotal de linhas extraídas:", len(base_pdf))
print("Meses que falharam:", falharam if falharam else "nenhum!")
print("\nFaixa de preço da cesta:")
print(base_pdf['valor'].describe()[['min', 'max']])
print("\nContagem de capitais por mês:")
print(base_pdf.groupby('aammes').size().value_counts().sort_index())


### 4.1 Correção manual do mês 2005/07

O boletim de julho/2005 não pôde ser extraído automaticamente pelo parser acima (formatação divergente no PDF original). Os valores desse mês foram, portanto, **transcritos manualmente** a partir do PDF oficial do DIEESE e são inseridos diretamente aqui, no mesmo formato da base extraída automaticamente.


In [ ]:
linha_200507 = [
    {"capital": "Natal", "var_mensal": 0.36, "valor": 140.25, "pct_sal": 50.62, "tempo": "102h51m", "var_ano": 6.41, "var_12": -0.15},
    {"capital": "João Pessoa", "var_mensal": -0.09, "valor": 143.91, "pct_sal": 51.94, "tempo": "105h32m", "var_ano": 14.10, "var_12": 2.25},
    {"capital": "Florianópolis", "var_mensal": -1.23, "valor": 163.81, "pct_sal": 59.13, "tempo": "120h08m", "var_ano": 4.06, "var_12": 0.79},
    {"capital": "Vitória", "var_mensal": -1.47, "valor": 160.50, "pct_sal": 57.93, "tempo": "117h42m", "var_ano": 5.33, "var_12": 4.63},
    {"capital": "Belo Horizonte", "var_mensal": -1.90, "valor": 164.87, "pct_sal": 59.51, "tempo": "120h54m", "var_ano": 8.27, "var_12": -2.78},
    {"capital": "Salvador", "var_mensal": -1.99, "valor": 134.23, "pct_sal": 48.45, "tempo": "98h26m", "var_ano": 6.67, "var_12": -1.29},
    {"capital": "Rio de Janeiro", "var_mensal": -2.12, "valor": 168.60, "pct_sal": 60.86, "tempo": "123h38m", "var_ano": 1.95, "var_12": 0.04},
    {"capital": "São Paulo", "var_mensal": -2.69, "valor": 178.22, "pct_sal": 64.33, "tempo": "130h42m", "var_ano": 3.50, "var_12": 2.45},
    {"capital": "Belém", "var_mensal": -2.86, "valor": 149.46, "pct_sal": 53.95, "tempo": "109h36m", "var_ano": -0.16, "var_12": -2.65},
    {"capital": "Goiânia", "var_mensal": -3.02, "valor": 152.68, "pct_sal": 55.11, "tempo": "111h58m", "var_ano": 2.55, "var_12": 3.78},
    {"capital": "Curitiba", "var_mensal": -3.08, "valor": 163.21, "pct_sal": 58.91, "tempo": "119h41m", "var_ano": 4.68, "var_12": -1.37},
    {"capital": "Fortaleza", "var_mensal": -3.38, "valor": 140.29, "pct_sal": 50.64, "tempo": "102h53m", "var_ano": 12.47, "var_12": -2.70},
    {"capital": "Recife", "var_mensal": -3.40, "valor": 143.40, "pct_sal": 51.76, "tempo": "105h10m", "var_ano": 16.59, "var_12": 1.29},
    {"capital": "Porto Alegre", "var_mensal": -4.01, "valor": 174.75, "pct_sal": 63.08, "tempo": "128h09m", "var_ano": 0.00, "var_12": -3.89},
    {"capital": "Brasília", "var_mensal": -4.55, "valor": 165.14, "pct_sal": 59.61, "tempo": "121h06m", "var_ano": -2.13, "var_12": 0.07},
    {"capital": "Aracaju", "var_mensal": -5.07, "valor": 139.92, "pct_sal": 50.50, "tempo": "102h36m", "var_ano": 6.56, "var_12": 1.13},
]
df_200507 = pd.DataFrame(linha_200507)
df_200507["aammes"] = "200507"

# junta a correção manual com a base extraída automaticamente
base_pdf = pd.concat([base_pdf, df_200507], ignore_index=True)
print("Total final:", len(base_pdf), "linhas,", base_pdf['aammes'].nunique(), "meses")


## 5. Base tratada final

Por fim, a base integrada e limpa é exportada para `base_tratada_dieese_completa.csv`. Este é o arquivo que compõe o entregável **"Base tratada"** do projeto (junto com os dados brutos em `dados_brutos/`, o log de proveniência e a dataset card).


In [ ]:
base_pdf.to_csv("base_tratada_dieese_completa.csv", index=False, encoding="utf-8-sig")
print("Salvo em base_tratada_dieese_completa.csv")


### Conferência final

Recarrega o CSV salvo para confirmar que a exportação funcionou corretamente e dar uma visão geral da base final.

In [ ]:
conferencia = pd.read_csv("base_tratada_dieese_completa.csv")
print(conferencia.info())
conferencia.head()
